# Colab launcher

Runs the heavy experiments on a Colab GPU. This notebook holds no project logic: it sets up the
VM and calls `python -m experiments.run`, exactly as on a local machine.

**Runtime → Change runtime type:** A100 for full fine-tunes; L4 is fine for the probe, feature
extraction and smoke runs. Avoid T4 (no bfloat16, which `finetune.amp_dtype` uses) and TPUs (the
code is PyTorch/CUDA).

**Colab secrets (key icon, left sidebar), with notebook access on:**
- `GH_TOKEN`: GitHub token with read access, only needed while the repo is private.
- `HF_TOKEN`: Hugging Face token; DINOv3 checkpoints are gated, so accept the licence on the hub first.

Data, runs and the Hugging Face cache persist in `MyDrive/geo-relations/`; everything else is
rebuilt each session.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/geo-relations/data/raw /content/drive/MyDrive/geo-relations/runs /content/drive/MyDrive/geo-relations/hf_cache

In [ ]:
import os
from google.colab import userdata

REPO = "parsa-hke/veridiagram"  # GitHub owner/name
BRANCH = "main"
WORKDIR = "/content/veridiagram"

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret("GH_TOKEN")
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.isdir(f"{WORKDIR}/.git"):
    !git -C {WORKDIR} pull --ff-only
else:
    !git clone -q -b {BRANCH} {url} {WORKDIR}
    !git -C {WORKDIR} remote set-url origin https://github.com/{REPO}.git  # drop token from .git/config
%cd {WORKDIR}
!git log -1 --oneline

In [ ]:
# Installs the pinned versions over Colab's preinstalled ones. If pip asks for a restart,
# use Runtime -> Restart session, then rerun the cells above (they are idempotent) and continue.
!pip install -q -r requirements.txt

In [ ]:
os.environ["HF_HOME"] = "/content/drive/MyDrive/geo-relations/hf_cache"  # backbone weights survive sessions
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
!python -m experiments.run --device-info

## Smoke run (must pass before any full run)

In [ ]:
!python -m experiments.run --config configs/probe.yaml --smoke --overlay configs/colab.yaml
!python -m experiments.run --config configs/finetune.yaml --smoke --overlay configs/colab.yaml

## Full run

Results go to `MyDrive/geo-relations/runs/<name>_s<seed>/` and `MyDrive/geo-relations/runs/log.csv`.

In [ ]:
CONFIG = "configs/finetune.yaml"
SEEDS = [0, 1, 2]
for seed in SEEDS:
    !python -m experiments.run --config {CONFIG} --overlay configs/colab.yaml --set seed={seed}